# Notebook 05 - Planificateur de Repas : Data Fusion LINQ + Theoreme Hierarchique

**Navigation** : [Index Z3](./README.md) | [Index SymbolicAI](../../README.md) | [Index general](../../../README.md)

## Objectifs d'apprentissage

- Comprendre le **theoremee hierarchique** : modeles Z3 avec objets imbriques
- Pratiquer la **fusion de donnees** entre sources externes (nutrition) et variables Z3
- Modeliser un **planificateur de repas** equilibre avec contraintes nutritionnelles
- Explorer l'**optimisation** (minimiser un cout sous contraintes)

## Prerequis

- Notebooks 01 (Introduction Z3.Linq) et 04 (Nested Arrays)
- Concepts : types generiques C#, LINQ, contraintes lineaires

**Duree estimee** : 30-40 minutes

In [1]:
#r "nuget: Z3.Linq"
using Z3.Linq;
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;
Console.WriteLine("Imports OK : Z3.Linq, Microsoft.Z3, System.Linq");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Z3.Linq, 2.0.1

Imports OK : Z3.Linq, Microsoft.Z3, System.Linq


---

## 1. Contexte : Le theoreme hierarchique

Dans les notebooks precedents, nous avons utilise `Symbols<T1, T2, ...>` pour creer des variables Z3 plates. Le **theoreme hierarchique** generalise cette approche :

- Un objet parent (ex: `Meal`) contient des references vers des objets enfants (ex: `Course`)
- Chaque niveau de la hierarchie possede ses propres contraintes
- Le solveur Z3 resout le systeme complet de maniere coherente

### Analogie

```
Meal (contraintes globales : calories totales, budget)
  |-- Entree (contraintes : type, taille)
  |-- Plat principal (contraintes : proteines minimales)
  |-- Dessert (contraintes : sucre maximal)
```

La **fusion de donnees** consiste a injecter des donnees externes (table nutritionnelle) dans les contraintes du solveur.

## 2. Base de donnees nutritionnelle (data fusion)

Nous definissons une base de donnees d'aliments avec leurs proprietes nutritionnelles. Ces donnees seront utilisees par le solveur pour selectionner des combinaisons valides.

In [2]:
// Base de donnees nutritionnelle : fusion de donnees statiques avec le solveur
public class FoodItem
{
    public string Name { get; set; }
    public int Calories { get; set; }      // kcal par portion
    public int Protein { get; set; }       // grammes
    public int Carbs { get; set; }         // grammes
    public int Fat { get; set; }           // grammes
    public int Cost { get; set; }          // prix en centimes
    public string Category { get; set; }   // entree, plat, dessert
}

var foodDatabase = new List<FoodItem>
{
    // Entrees
    new FoodItem { Name = "Salade verte",    Calories = 80,  Protein = 2,  Carbs = 8,   Fat = 4,  Cost = 300, Category = "entree" },
    new FoodItem { Name = "Soupe de legumes", Calories = 120, Protein = 4,  Carbs = 18,  Fat = 3,  Cost = 250, Category = "entree" },
    new FoodItem { Name = "Carpaccio",        Calories = 150, Protein = 15, Carbs = 2,   Fat = 8,  Cost = 600, Category = "entree" },
    // Plats principaux
    new FoodItem { Name = "Poulet roti",      Calories = 350, Protein = 30, Carbs = 5,   Fat = 20, Cost = 500, Category = "plat" },
    new FoodItem { Name = "Saumon grille",    Calories = 300, Protein = 28, Carbs = 0,   Fat = 18, Cost = 800, Category = "plat" },
    new FoodItem { Name = "Pates bolognaise", Calories = 450, Protein = 18, Carbs = 55,  Fat = 14, Cost = 350, Category = "plat" },
    new FoodItem { Name = "Risotto",          Calories = 400, Protein = 10, Carbs = 60,  Fat = 12, Cost = 400, Category = "plat" },
    // Desserts
    new FoodItem { Name = "Fruit frais",      Calories = 70,  Protein = 1,  Carbs = 15,  Fat = 0,  Cost = 200, Category = "dessert" },
    new FoodItem { Name = "Mousse au chocolat",Calories = 250, Protein = 4,  Carbs = 28,  Fat = 14, Cost = 350, Category = "dessert" },
    new FoodItem { Name = "Yaourt nature",    Calories = 60,  Protein = 5,  Carbs = 6,   Fat = 2,  Cost = 150, Category = "dessert" },
};

Console.WriteLine($"Base de donnees chargee : {foodDatabase.Count} aliments");
Console.WriteLine(string.Join("\n", foodDatabase.GroupBy(f => f.Category)
    .Select(g => $"  {g.Key} : {g.Count()} items")));

Base de donnees chargee : 10 aliments


  entree : 3 items
  plat : 4 items
  dessert : 3 items


### Interpretation

La base contient 10 aliments repartis en 3 categories (entree, plat, dessert). Chaque aliment a des proprietes nutritionnelles et un cout. Le solveur devra selectionner exactement **1 entree + 1 plat + 1 dessert** qui satisfont les contraintes globales.

## 3. Approche 1 : Selection par index avec contraintes lineaires

La premiere approche utilise des variables entieres pour selectionner un aliment dans chaque categorie. Les contraintes sont exprimees lineairement en fonction de l'index.

In [3]:
// Approche index-based : chaque variable represente l'index d'un aliment
// Les contraintes Z3.Linq doivent utiliser des expressions lineaires sur les variables

var entrees = foodDatabase.Where(f => f.Category == "entree").ToList();
var plats = foodDatabase.Where(f => f.Category == "plat").ToList();
var desserts = foodDatabase.Where(f => f.Category == "dessert").ToList();

using (var ctx = new Z3Context())
{
    // Variables : index de l'aliment choisi dans chaque categorie
    // Contraintes de bornes avec expressions lineaires (pattern Z3.Linq)
    var theorem = from meal in ctx.NewTheorem<Symbols<int, int, int>>()
        where meal.X1 >= 0 && meal.X1 - 3 <= 0   // 0 <= X1 <= 2 (3 entrees)
        where meal.X2 >= 0 && meal.X2 - 4 <= 0   // 0 <= X2 <= 3 (4 plats)
        where meal.X3 >= 0 && meal.X3 - 3 <= 0   // 0 <= X3 <= 2 (3 desserts)
        select meal;

    var solution = theorem.Solve();
    
    if (solution != null)
    {
        var e = entrees[solution.X1];
        var p = plats[solution.X2];
        var d = desserts[solution.X3];
        int totalCal = e.Calories + p.Calories + d.Calories;
        int totalCost = e.Cost + p.Cost + d.Cost;
        
        Console.WriteLine("Menu trouve (sans contraintes nutritionnelles) :");
        Console.WriteLine($"  Entree  : {e.Name} ({e.Calories} kcal)");
        Console.WriteLine($"  Plat    : {p.Name} ({p.Calories} kcal)");
        Console.WriteLine($"  Dessert : {d.Name} ({d.Calories} kcal)");
        Console.WriteLine($"  Total   : {totalCal} kcal, {totalCost/100.0:F2} euros");
    }
    else
    {
        Console.WriteLine("Aucune solution trouvee");
    }
}

Menu trouve (sans contraintes nutritionnelles) :


  Entree  : Salade verte (80 kcal)


  Plat    : Poulet roti (350 kcal)


  Dessert : Fruit frais (70 kcal)


  Total   : 500 kcal, 10,00 euros


### Interpretation

Sans contraintes nutritionnelles, le solveur trouve la premiere combinaison valide (index 0, 0, 0). Le resultat est trivialement la premiere entree, le premier plat et le premier dessert de la base. Nous devons ajouter des contraintes pour obtenir un menu equilibre.

## 4. Approche 2 : Contraintes nutritionnelles avec enumeration

La contrainte Z3.Linq fonctionne avec des expressions lineaires. Pour injecter les donnees nutritionnelles, nous enumerons les combinaisons possibles et posons des contraintes sur les totaux.

**Strategie** : generer toutes les combinaisons possibles, puis filtrer avec Z3 sur les contraintes globales (calories, proteines, budget).

In [4]:
// Approche par enumeration + filtrage Z3
// Objectif : trouver un menu avec 600-900 kcal, >= 25g proteines, cout <= 12 euros

int minCalories = 600, maxCalories = 900;
int minProtein = 25;
int maxCostCents = 1200; // 12 euros

var validMenus = new List<string>();

// Enumeration des combinaisons + verification des contraintes
foreach (var e in entrees)
    foreach (var p in plats)
        foreach (var d in desserts)
        {
            int cal = e.Calories + p.Calories + d.Calories;
            int prot = e.Protein + p.Protein + d.Protein;
            int cost = e.Cost + p.Cost + d.Cost;
            
            if (cal >= minCalories && cal <= maxCalories 
                && prot >= minProtein && cost <= maxCostCents)
            {
                validMenus.Add($"{e.Name} + {p.Name} + {d.Name} = {cal} kcal, {prot}g prot, {cost/100.0:F2} EUR");
            }
        }

Console.WriteLine($"Menus valides (600-900 kcal, >= 25g prot, <= 12 EUR) : {validMenus.Count}/{entrees.Count * plats.Count * desserts.Count} combinaisons");
Console.WriteLine();
foreach (var menu in validMenus.Take(8))
    Console.WriteLine($"  {menu}");
if (validMenus.Count > 8)
    Console.WriteLine($"  ... et {validMenus.Count - 8} autres");

Menus valides (600-900 kcal, >= 25g prot, <= 12 EUR) : 8/36 combinaisons


  Salade verte + Poulet roti + Mousse au chocolat = 680 kcal, 36g prot, 11,50 EUR


  Soupe de legumes + Poulet roti + Mousse au chocolat = 720 kcal, 38g prot, 11,00 EUR


  Soupe de legumes + Pates bolognaise + Mousse au chocolat = 820 kcal, 26g prot, 9,50 EUR


  Soupe de legumes + Pates bolognaise + Yaourt nature = 630 kcal, 27g prot, 7,50 EUR


  Carpaccio + Pates bolognaise + Fruit frais = 670 kcal, 34g prot, 11,50 EUR


  Carpaccio + Pates bolognaise + Yaourt nature = 660 kcal, 38g prot, 11,00 EUR


  Carpaccio + Risotto + Fruit frais = 620 kcal, 26g prot, 12,00 EUR


  Carpaccio + Risotto + Yaourt nature = 610 kcal, 30g prot, 11,50 EUR


### Interpretation

L'enumeration exhaustive fonctionne car l'espace est petit (3 x 4 x 3 = 36 combinaisons). Pour de grands catalogues, le solveur Z3 est preferable car il explore intelligemment l'espace de recherche plutot que de tout enumerer.

## 5. Approche 3 : Modele Z3 avec variables booleennes (selection)

Pour modeliser la selection d'aliments directement dans Z3, nous utilisons des **variables booleennes** : chaque aliment est soit selectionne (`true`) soit non selectionne (`false`).

### Principe

- Une variable booleenne par aliment
- Exactement 1 entree + 1 plat + 1 dessert selectionnes
- Contraintes nutritionnelles sur la somme ponderee

In [5]:
// Approche booleenne Z3 : chaque aliment est une variable bool
// Avantage : le solveur explore l'espace, pas d'enumeration

using (var z3ctx = new Context())
{
    var solver = z3ctx.MkSolver();
    
    // Variables booleennes : une par aliment
    var selections = foodDatabase
        .Select((food, i) => new { Food = food, Var = (BoolExpr)z3ctx.MkConst($"sel_{i}_{food.Name}", z3ctx.BoolSort) })
        .ToList();
    
    // Contrainte 1 : exactement 1 entree selectionnee
    var entreeVars = selections.Where(s => s.Food.Category == "entree").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(entreeVars));           // au moins 1
    foreach (var v1 in entreeVars)                 // au plus 1 (pairwise exclusion)
        foreach (var v2 in entreeVars)
            if (v1 != v2)
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    // Contrainte 2 : exactement 1 plat selectionne
    var platVars = selections.Where(s => s.Food.Category == "plat").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(platVars));
    foreach (var v1 in platVars)
        foreach (var v2 in platVars)
            if (v1 != v2)
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    // Contrainte 3 : exactement 1 dessert selectionne
    var dessertVars = selections.Where(s => s.Food.Category == "dessert").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(dessertVars));
    foreach (var v1 in dessertVars)
        foreach (var v2 in dessertVars)
            if (v1 != v2)
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    // Contraintes nutritionnelles lineaires (implication conditionnelle)
    IntExpr totalCal = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Calories), z3ctx.MkInt(0))));
    
    IntExpr totalProt = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Protein), z3ctx.MkInt(0))));
    
    IntExpr totalCost = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Cost), z3ctx.MkInt(0))));
    
    solver.Add(z3ctx.MkGe(totalCal, z3ctx.MkInt(600)));
    solver.Add(z3ctx.MkLe(totalCal, z3ctx.MkInt(900)));
    solver.Add(z3ctx.MkGe(totalProt, z3ctx.MkInt(25)));
    solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(1200)));
    
    Console.WriteLine($"Contraintes : {solver.NumAssertions} assertions");
    
    var result = solver.Check();
    Console.WriteLine($"Resultat solve : {result}");
    
    if (result == Status.SATISFIABLE)
    {
        var model = solver.Model;
        Console.WriteLine("\nMenu equilibre trouve :");
        int cal = 0, prot = 0, cost = 0;
        foreach (var s in selections)
        {
            var val = model.Eval(s.Var, true);
            if (val.BoolValue == Z3_lbool.Z3_L_TRUE)
            {
                Console.WriteLine($"  [{s.Food.Category}] {s.Food.Name} : {s.Food.Calories} kcal, {s.Food.Protein}g prot, {s.Food.Cost/100.0:F2} EUR");
                cal += s.Food.Calories;
                prot += s.Food.Protein;
                cost += s.Food.Cost;
            }
        }
        Console.WriteLine($"  --- Total : {cal} kcal, {prot}g proteines, {cost/100.0:F2} EUR ---");
    }
    else
    {
        Console.WriteLine("Aucun menu ne satisfait les contraintes.");
    }
}

Contraintes : 31 assertions


Resultat solve : SATISFIABLE



Menu equilibre trouve :


  [entree] Soupe de legumes : 120 kcal, 4g prot, 2,50 EUR


  [plat] Pates bolognaise : 450 kcal, 18g prot, 3,50 EUR


  [dessert] Mousse au chocolat : 250 kcal, 4g prot, 3,50 EUR


  --- Total : 820 kcal, 26g proteines, 9,50 EUR ---


### Interpretation

L'approche booleenne utilise **`MkITE` (If-Then-Else)** pour conditionnellement ajouter la contribution nutritionnelle de chaque aliment. C'est la cle technique de la **data fusion** : les donnees statiques (calories, proteines, cout) sont injectees dans les expressions Z3 via `MkITE`.

| Technique | Role |
|-----------|------|
| `BoolExpr` par aliment | Selection binaire |
| `MkOr` + exclusion mutuelle | Exactement 1 par categorie |
| `MkITE(sel, valeur, 0)` | Contribution conditionnelle |
| Somme des `MkITE` | Total nutritionnel global |

## 6. Optimisation : menu le moins cher (dichotomie)

Le solveur Z3 (`MkSolver`) trouve une solution satisfaisante, mais pas forcement optimale. Pour trouver le menu **le moins cher**, nous utilisons une **recherche par dichotomie** :

1. On fixe une borne superieure sur le cout
2. On demande au solveur s'il existe un menu dans ce budget
3. Si oui, on essaie un budget plus faible ; si non, on augmente
4. On repete jusqu'a trouver le minimum exact

Cette technique (binary search on objective) est un pattern classique en optimisation par solveur SAT/SMT.

In [6]:
// Optimisation : trouver le menu equilibre le MOINS CHER
// On utilise le solver avec exploration iterative : on ajoute une contrainte
// de cout de plus en plus strict jusqu'a ce que le probleme devienne UNSAT

using (var z3ctx = new Context())
{
    var solver = z3ctx.MkSolver();
    
    // Variables booleennes
    var selections = foodDatabase
        .Select((food, i) => new { Food = food, Var = (BoolExpr)z3ctx.MkConst($"sel_{i}_{food.Name}", z3ctx.BoolSort) })
        .ToList();
    
    // Exactement 1 par categorie
    var entreeVars = selections.Where(s => s.Food.Category == "entree").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(entreeVars));
    foreach (var v1 in entreeVars)
        foreach (var v2 in entreeVars)
            if (!v1.Equals(v2))
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    var platVars = selections.Where(s => s.Food.Category == "plat").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(platVars));
    foreach (var v1 in platVars)
        foreach (var v2 in platVars)
            if (!v1.Equals(v2))
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    var dessertVars = selections.Where(s => s.Food.Category == "dessert").Select(s => (BoolExpr)s.Var).ToArray();
    solver.Add(z3ctx.MkOr(dessertVars));
    foreach (var v1 in dessertVars)
        foreach (var v2 in dessertVars)
            if (!v1.Equals(v2))
                solver.Add(z3ctx.MkImplies(v1, z3ctx.MkNot(v2)));
    
    // Contraintes nutritionnelles
    IntExpr totalCal = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Calories), z3ctx.MkInt(0))));
    IntExpr totalProt = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Protein), z3ctx.MkInt(0))));
    IntExpr totalCost = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0), 
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Food.Cost), z3ctx.MkInt(0))));
    
    solver.Add(z3ctx.MkGe(totalCal, z3ctx.MkInt(600)));
    solver.Add(z3ctx.MkLe(totalCal, z3ctx.MkInt(900)));
    solver.Add(z3ctx.MkGe(totalProt, z3ctx.MkInt(25)));
    solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(1200)));
    
    // Recherche du cout minimum par dichotomie
    int lo = 0, hi = 1200, bestCost = -1;
    Status bestResult = Status.UNSATISFIABLE;
    
    while (lo <= hi)
    {
        solver.Push();
        int mid = (lo + hi) / 2;
        solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(mid)));
        var r = solver.Check();
        solver.Pop();
        
        if (r == Status.SATISFIABLE)
        {
            bestCost = mid;
            bestResult = r;
            hi = mid - 1;  // essayer moins cher
        }
        else
        {
            lo = mid + 1;  // besoin de plus de budget
        }
    }
    
    // Resoudre avec le meilleur cout trouve
    solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(bestCost)));
    var result = solver.Check();
    Console.WriteLine($"Optimisation (dichotomie) : cout minimum = {bestCost/100.0:F2} EUR, resultat = {result}");
    
    if (result == Status.SATISFIABLE)
    {
        var model = solver.Model;
        Console.WriteLine("\nMenu optimal (cout minimum) :");
        int cal = 0, prot = 0, cost = 0;
        foreach (var s in selections)
        {
            var val = model.Eval(s.Var, true);
            if (val.BoolValue == Z3_lbool.Z3_L_TRUE)
            {
                Console.WriteLine($"  [{s.Food.Category}] {s.Food.Name} : {s.Food.Calories} kcal, {s.Food.Protein}g prot, {s.Food.Cost/100.0:F2} EUR");
                cal += s.Food.Calories;
                prot += s.Food.Protein;
                cost += s.Food.Cost;
            }
        }
        Console.WriteLine($"  === Total : {cal} kcal, {prot}g proteines, {cost/100.0:F2} EUR ===");
    }
    else
    {
        Console.WriteLine("Aucun menu optimal trouve.");
    }
}

Optimisation (dichotomie) : cout minimum = 7,50 EUR, resultat = SATISFIABLE



Menu optimal (cout minimum) :


  [entree] Soupe de legumes : 120 kcal, 4g prot, 2,50 EUR


  [plat] Pates bolognaise : 450 kcal, 18g prot, 3,50 EUR


  [dessert] Yaourt nature : 60 kcal, 5g prot, 1,50 EUR


  === Total : 630 kcal, 27g proteines, 7,50 EUR ===


### Interpretation

La **dichotomie sur le cout** est un pattern classique en optimisation SAT/SMT. Plutot que d'utiliser `MkOptimizer` (qui peut etre instable dans certains bindings), on utilise `solver.Push()`/`solver.Pop()` pour tester differentes bornes de cout sans reconstruire le solver a chaque fois.

| Aspect | Solve simple | Dichotomie |
|--------|-------------|------------|
| Objectif | Premiere solution | Solution optimale |
| Cout | Arbitraire | Minimum garanti |
| Appels solver | 1 | O(log(budget_max)) |
| Stabilite | Toujours | Toujours |

## 7. Tableau recapitulatif des approches

| Approche | Technique | Avantage | Limite |
|----------|-----------|----------|--------|
| Index Z3.Linq | `Symbols<int,int,int>` | Syntaxe LINQ naturelle | Contraintes index-only |
| Enumeration | Boucles C# + filtres | Simple a comprendre | Pas scalable (O(n^3)) |
| Booleenne Z3 | `BoolExpr` + `MkITE` | Data fusion native | Syntaxe bas niveau |
| Dichotomie | `Push/Pop` + binary search | Optimisation stable | O(log n) appels solver |

---

## Exercices

Les exercices suivants vous demandent d'etendre le modele du planificateur de repas.

### Exercice 1 : Menu equilibre avec contrainte de glucides

Ajoutez une contrainte sur les **glucides totaux** (entre 40g et 80g) au modele d'optimisation (section 6). Le menu optimal doit toujours minimiser le cout, mais satisfaire cette contrainte supplementaire.

**Indice** : Ajoutez un `IntExpr totalCarbs` avec le meme pattern `MkITE` que `totalCal` et `totalProt`, puis ajoutez les contraintes `MkGe` / `MkLe`.

In [7]:
// Exercice 1 : ajoutez la contrainte 40g <= glucides <= 80g au modele d'optimisation
// Reprenez le code de la section 6 et ajoutez totalCarbs

// TODO etudiant : implementer la contrainte de glucides
// int minCarbs = 40, maxCarbs = 80;
// IntExpr totalCarbs = ...
// optimizer.Add(z3ctx.MkGe(totalCarbs, z3ctx.MkInt(minCarbs)));
// optimizer.Add(z3ctx.MkLe(totalCarbs, z3ctx.MkInt(maxCarbs)));

Console.WriteLine("Exercice a completer : contrainte de glucides");

Exercice a completer : contrainte de glucides


### Exercice 2 : Planificateur de repas multi-jours

Etendez le modele pour planifier un menu sur **2 jours** (6 repas : 2 entrees, 2 plats, 2 desserts) sans repetition. Contrainte : chaque aliment ne peut etre selectionne qu'**au maximum 1 fois**.

**Etape 1** : Creez 6 groupes de variables booleennes (ou utilisez des paires jour/categorie).

**Etape 2** : Ajoutez la contrainte de non-repetition (un aliment ne peut pas etre vrai dans 2 groupes differents).

**Indice** : Pour chaque aliment, la somme de ses apparitions dans les differents repas doit etre <= 1.

In [8]:
// Exercice 2 : planificateur multi-jours sans repetition

// TODO etudiant : creer les variables pour 2 jours de menus
// Indice : utilisez un dictionnaire (jour, categorie) -> BoolExpr[]
// Pour la non-repetition : pour chaque aliment, sum(apparitions) <= 1

Console.WriteLine("Exercice a completer : planificateur multi-jours");

Exercice a completer : planificateur multi-jours


### Exercice 3 : Ajout d'un plat supplementaire (boisson)

Ajoutez une categorie **"boisson"** a la base de donnees (3 boissons) et modifiez le modele d'optimisation pour inclure exactement 1 boisson par repas. Verifiez que les contraintes nutritionnelles sont toujours satisfaites.

**Etape 1** : Ajoutez 3 boissons a `foodDatabase`.

**Etape 2** : Ajoutez le groupe "boisson" dans les contraintes `ExactlyOne`.

**Etape 3** : Re-executez l'optimisation et comparez le nouveau menu optimal.

In [9]:
// Exercice 3 : ajout d'une categorie boisson

// TODO etudiant : ajouter 3 boissons a foodDatabase
// Exemples : eau (0 kcal), jus d'orange (110 kcal, 25g glucides), vin (85 kcal)
// TODO etudiant : ajouter la contrainte ExactlyOne pour la categorie boisson

Console.WriteLine("Exercice a completer : ajout de boissons");

Exercice a completer : ajout de boissons


---

## Conclusion

Ce notebook a explore la **modelisation hierarchique** avec Z3 :

1. **Data fusion** : injection de donnees nutritionnelles dans les expressions Z3 via `MkITE`
2. **Variables booleennes** : selection d'aliments avec contraintes de cardinalite (exactement 1 par categorie)
3. **Optimisation par dichotomie** : `Push/Pop` + binary search pour trouver le menu le moins cher
4. **Comparaison des approches** : LINQ index, enumeration, modele booleen, dichotomie

### Points cles a retenir

| Concept | Implementation Z3 |
|---------|-------------------|
| Selection binaire | `BoolExpr` + exclusion mutuelle |
| Data fusion | `MkITE(variable, valeur, 0)` |
| Contrainte de somme | Agregation `MkAdd` des `MkITE` |
| Optimisation | Dichotomie avec `Push/Pop` |

**Navigation** : [Index Z3](./README.md) | [Notebook precedent (Nested Arrays)](04_Nested_Arrays_2D.ipynb)